# Layout and mosaic

Port of `packages/niivue/examples/layout.html`. Demonstrates multiplanar layout controls, mosaic strings, hero fraction, margins, ruler, and radiological display.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"


def hex_to_rgba(value):
    value = value.lstrip("#")
    return [int(value[i : i + 2], 16) / 255 for i in (0, 2, 4)] + [1.0]


nv = NiiVue(
    background_color=[0.2, 0.2, 0.2, 1.0],
    is_colorbar_visible=True,
    slice_type=3,
    backend="webgl2",
)

slice_type = widgets.Dropdown(
    options=[
        ("Axial", 0),
        ("Coronal", 1),
        ("Sagittal", 2),
        ("A+C+S+R", 3),
        ("Render", 4),
        ("Mosaic", 5),
    ],
    value=3,
    description="Slice",
)
multiplanar_type = widgets.Dropdown(
    options=[("Auto", 0), ("Column", 1), ("Grid", 2), ("Row", 3)], value=0, description="Layout"
)
hero = widgets.IntSlider(value=0, min=0, max=8, step=1, description="Hero")
force_render = widgets.Checkbox(value=False, description="Force render")
mosaic = widgets.Text(value="A -20 50 60 70 L C -10 -20 -50 S R X 0 R X -0", description="Mosaic")
margin = widgets.IntSlider(value=0, min=0, max=8, step=1, description="Margin")
background = widgets.ColorPicker(value="#333334", description="Background")
equal_size = widgets.Checkbox(value=False, description="Equal size")
clip_dark = widgets.Checkbox(value=False, description="Clip dark")
radiological = widgets.Checkbox(value=False, description="Radiological")
ruler = widgets.Checkbox(value=False, description="Ruler")


def update_slice(change=None):
    if slice_type.value == 5:
        nv.mosaic_string = mosaic.value
    else:
        nv.mosaic_string = ""
        nv.slice_type = slice_type.value


def update_multiplanar(change):
    nv.multiplanar_type = change["new"]


def update_hero(change):
    nv.hero_fraction = change["new"] * 0.1


def update_force_render(change):
    nv.show_render = 1 if change["new"] else 0


def update_mosaic(change):
    if slice_type.value == 5:
        nv.mosaic_string = change["new"]


def update_margin(change):
    nv.tile_margin = change["new"] * 2


def update_background(change):
    nv.background_color = hex_to_rgba(change["new"])


def update_equal_size(change):
    nv.is_equal_size = change["new"]


def update_clip_dark(change):
    nv.volume_is_alpha_clip_dark = change["new"]


def update_radiological(change):
    nv.is_radiological = change["new"]


def update_ruler(change):
    nv.is_ruler_visible = change["new"]


slice_type.observe(update_slice, names="value")
multiplanar_type.observe(update_multiplanar, names="value")
hero.observe(update_hero, names="value")
force_render.observe(update_force_render, names="value")
mosaic.observe(update_mosaic, names="value")
margin.observe(update_margin, names="value")
background.observe(update_background, names="value")
equal_size.observe(update_equal_size, names="value")
clip_dark.observe(update_clip_dark, names="value")
radiological.observe(update_radiological, names="value")
ruler.observe(update_ruler, names="value")

controls = widgets.VBox(
    [
        widgets.HBox([slice_type, multiplanar_type, hero, force_render]),
        widgets.HBox([margin, background, equal_size, clip_dark, radiological, ruler]),
        mosaic,
    ]
)
display(widgets.VBox([controls, nv]))

nv.load_volumes([{"url": f"{VOLUMES}/mni152.nii.gz"}])
nv.load_meshes([{"url": f"{MESHES}/dpsv.trx"}])
update_slice()